In [1]:
import hashlib
import json
import sqlite3
import uuid
from time import time

In [2]:
class Blockchain:
    def __init__(self):
        self.chain = []
        self.pending_transactions = []
        self.create_block(proof=1, previous_hash='0')  # Genesis block

    def create_block(self, proof, previous_hash):
        block = {
            'index': len(self.chain) + 1,
            'timestamp': time(),
            'transactions': self.pending_transactions,
            'proof': proof,
            'previous_hash': previous_hash
        }
        self.pending_transactions = []
        self.chain.append(block)
        return block

    def get_previous_block(self):
        return self.chain[-1]

    def proof_of_work(self, previous_proof):
        new_proof = 1
        check_proof = False

        while not check_proof:
            hash_operation = hashlib.sha256(
                str(new_proof**2 - previous_proof**2).encode()
            ).hexdigest()

            if hash_operation[:4] == '0000':
                check_proof = True
            else:
                new_proof += 1

        return new_proof

    def hash(self, block):
        encoded_block = json.dumps(block, sort_keys=True).encode()
        return hashlib.sha256(encoded_block).hexdigest()

    def add_transaction(self, sender, receiver, amount):
        self.pending_transactions.append({
            'sender': sender,
            'receiver': receiver,
            'amount': amount
        })

    def is_chain_valid(self):
        previous_block = self.chain[0]
        block_index = 1

        while block_index < len(self.chain):
            block = self.chain[block_index]

            if block['previous_hash'] != self.hash(previous_block):
                return False

            previous_proof = previous_block['proof']
            proof = block['proof']

            hash_operation = hashlib.sha256(
                str(proof**2 - previous_proof**2).encode()
            ).hexdigest()

            if hash_operation[:4] != '0000':
                return False

            previous_block = block
            block_index += 1

        return True

In [4]:
blockchain = Blockchain()

print("Blockchain initialized successfully.")
print("Genesis Block:")
print(blockchain.chain[0])

Blockchain initialized successfully.
Genesis Block:
{'index': 1, 'timestamp': 1773732818.138859, 'transactions': [], 'proof': 1, 'previous_hash': '0'}


In [6]:
conn = sqlite3.connect("wallet.db")
cursor = conn.cursor()

print("Database connected successfully.")

Database connected successfully.


In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    wallet_address TEXT UNIQUE NOT NULL,
    balance REAL DEFAULT 100.0
)
""")

conn.commit()
print("Users table created successfully.")

Users table created successfully.


In [9]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sender TEXT NOT NULL,
    receiver TEXT NOT NULL,
    amount REAL NOT NULL,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
print("Transactions table created successfully.")

Transactions table created successfully.


In [10]:
print("System initialization completed.")
print("Blockchain length:", len(blockchain.chain))
print("Pending transactions:", blockchain.pending_transactions)

System initialization completed.
Blockchain length: 1
Pending transactions: []


In [11]:
def register_user(username, password):
    wallet_address = str(uuid.uuid4()).replace("-", "")[:16]
    default_balance = 100.0

    try:
        cursor.execute("""
        INSERT INTO users (username, password, wallet_address, balance)
        VALUES (?, ?, ?, ?)
        """, (username, password, wallet_address, default_balance))

        conn.commit()
        print("User registered successfully.")
        print("Username:", username)
        print("Wallet Address:", wallet_address)
        print("Balance:", default_balance)

    except sqlite3.IntegrityError:
        print("Username already exists. Try another username.")

In [12]:
register_user("varsha", "1234")

User registered successfully.
Username: varsha
Wallet Address: 7db5e3f63b894c63
Balance: 100.0


In [13]:
cursor.execute("SELECT * FROM users")
users = cursor.fetchall()

for user in users:
    print(user)

(1, 'varsha', '1234', '7db5e3f63b894c63', 100.0)


In [14]:
def login_user(username, password):
    hashed_password = hashlib.sha256(password.encode()).hexdigest()

    cursor.execute("""
    SELECT * FROM users
    WHERE username = ? AND password = ?
    """, (username, hashed_password))

    user = cursor.fetchone()

    if user:
        print("Login successful!")
        print("User ID:", user[0])
        print("Username:", user[1])
        print("Wallet Address:", user[3])
        print("Balance:", user[4])
        return user
    else:
        print("Invalid username or password.")
        return None

In [15]:
login_user("varsha", "1234")

Invalid username or password.


In [16]:
cursor.execute("SELECT username, password FROM users")
print(cursor.fetchall())

[('varsha', '1234')]


In [17]:
def login_user(username, password):
    cursor.execute("""
    SELECT * FROM users
    WHERE username = ? AND password = ?
    """, (username, password))

    user = cursor.fetchone()

    if user:
        print("Login successful!")
        print("Username:", user[1])
        print("Wallet:", user[3])
        print("Balance:", user[4])
        return user
    else:
        print("Invalid username or password.")
        return None

In [18]:
login_user("varsha", "1234")

Login successful!
Username: varsha
Wallet: 7db5e3f63b894c63
Balance: 100.0


(1, 'varsha', '1234', '7db5e3f63b894c63', 100.0)

In [19]:
def show_wallet_details(username):
    cursor.execute("""
    SELECT username, wallet_address, balance
    FROM users
    WHERE username = ?
    """, (username,))
    
    user = cursor.fetchone()
    
    if user:
        print("Wallet Details")
        print("Username      :", user[0])
        print("Wallet Address:", user[1])
        print("Balance       :", user[2], "Coins")
        return user
    else:
        print("User not found.")
        return None

In [20]:
show_wallet_details("varsha")

Wallet Details
Username      : varsha
Wallet Address: 7db5e3f63b894c63
Balance       : 100.0 Coins


('varsha', '7db5e3f63b894c63', 100.0)

In [21]:
def show_current_user_wallet(current_user):
    if current_user:
        print("Wallet Details")
        print("Username      :", current_user[1])
        print("Wallet Address:", current_user[3])
        print("Balance       :", current_user[4], "Coins")
    else:
        print("No user is currently logged in.")

In [22]:
show_current_user_wallet(current_user)

NameError: name 'current_user' is not defined

In [23]:
current_user = login_user("varsha", "1234")

Login successful!
Username: varsha
Wallet: 7db5e3f63b894c63
Balance: 100.0


In [24]:
show_current_user_wallet(current_user)

Wallet Details
Username      : varsha
Wallet Address: 7db5e3f63b894c63
Balance       : 100.0 Coins


In [25]:
def send_coins(sender_username, receiver_wallet, amount):
    try:
        amount = float(amount)
    except ValueError:
        print("Amount must be a number.")
        return

    if amount <= 0:
        print("Amount must be greater than 0.")
        return

    # Get sender details
    cursor.execute("""
    SELECT username, wallet_address, balance
    FROM users
    WHERE username = ?
    """, (sender_username,))
    sender = cursor.fetchone()

    if not sender:
        print("Sender not found.")
        return

    sender_wallet = sender[1]
    sender_balance = sender[2]

    # Get receiver details
    cursor.execute("""
    SELECT username, wallet_address, balance
    FROM users
    WHERE wallet_address = ?
    """, (receiver_wallet,))
    receiver = cursor.fetchone()

    if not receiver:
        print("Receiver wallet not found.")
        return

    receiver_balance = receiver[2]

    if sender_wallet == receiver_wallet:
        print("You cannot send coins to your own wallet.")
        return

    if sender_balance < amount:
        print("Insufficient balance.")
        return

    # Update balances
    new_sender_balance = sender_balance - amount
    new_receiver_balance = receiver_balance + amount

    cursor.execute("""
    UPDATE users
    SET balance = ?
    WHERE username = ?
    """, (new_sender_balance, sender_username))

    cursor.execute("""
    UPDATE users
    SET balance = ?
    WHERE wallet_address = ?
    """, (new_receiver_balance, receiver_wallet))

    # Record transaction in database
    cursor.execute("""
    INSERT INTO transactions (sender, receiver, amount)
    VALUES (?, ?, ?)
    """, (sender_wallet, receiver_wallet, amount))

    conn.commit()

    # Add transaction to blockchain pending list
    blockchain.add_transaction(sender_wallet, receiver_wallet, amount)

    print("Transaction successful!")
    print("Sender Wallet :", sender_wallet)
    print("Receiver Wallet:", receiver_wallet)
    print("Amount Sent   :", amount)
    print("New Balance   :", new_sender_balance)

In [26]:
register_user("rahul", "5678")

User registered successfully.
Username: rahul
Wallet Address: 28dc6f4c995d4fe9
Balance: 100.0


In [28]:
cursor.execute("SELECT username, wallet_address, balance FROM users")
users = cursor.fetchall()

for user in users:
    print(user)

('varsha', '7db5e3f63b894c63', 100.0)
('rahul', '28dc6f4c995d4fe9', 100.0)


In [29]:
send_coins("varsha", "PASTE_RAHUL_WALLET_ADDRESS_HERE", 25)

Receiver wallet not found.


In [30]:
cursor.execute("SELECT username, wallet_address, balance FROM users")
users = cursor.fetchall()

for user in users:
    print("Username:", user[0])
    print("Wallet :", user[1])
    print("Balance:", user[2])
    print("-------------------")

Username: varsha
Wallet : 7db5e3f63b894c63
Balance: 100.0
-------------------
Username: rahul
Wallet : 28dc6f4c995d4fe9
Balance: 100.0
-------------------


In [31]:
send_coins("varsha", "x9y8z7w6v5u4t3s2", 25)

Receiver wallet not found.


In [32]:
receiver_wallet = "PASTE_WALLET"
print("Receiver Wallet:", receiver_wallet)

Receiver Wallet: PASTE_WALLET


In [33]:
receiver_wallet = get_wallet_by_username("rahul")
send_coins("varsha", receiver_wallet, 25)

NameError: name 'get_wallet_by_username' is not defined

In [34]:
def get_wallet_by_username(username):
    cursor.execute("SELECT wallet_address FROM users WHERE username = ?", (username,))
    user = cursor.fetchone()
    if user:
        return user[0]
    else:
        return None

In [35]:
receiver_wallet = get_wallet_by_username("rahul")
print("Receiver Wallet:", receiver_wallet)

send_coins("varsha", receiver_wallet, 25)

Receiver Wallet: 28dc6f4c995d4fe9
Transaction successful!
Sender Wallet : 7db5e3f63b894c63
Receiver Wallet: 28dc6f4c995d4fe9
Amount Sent   : 25.0
New Balance   : 75.0


In [36]:
def mine_block():
    previous_block = blockchain.get_previous_block()
    previous_proof = previous_block['proof']
    proof = blockchain.proof_of_work(previous_proof)
    previous_hash = blockchain.hash(previous_block)
    block = blockchain.create_block(proof, previous_hash)

    print("Block mined successfully!")
    print("Block Index     :", block['index'])
    print("Timestamp       :", block['timestamp'])
    print("Transactions    :", block['transactions'])
    print("Proof           :", block['proof'])
    print("Previous Hash   :", block['previous_hash'])

In [37]:
mine_block()

Block mined successfully!
Block Index     : 2
Timestamp       : 1773733750.3614678
Transactions    : [{'sender': '7db5e3f63b894c63', 'receiver': '28dc6f4c995d4fe9', 'amount': 25.0}]
Proof           : 533
Previous Hash   : b4992cec8d1c2e39288f3051c177a689f8c172bc9b16865cc8c498731e52005a


In [38]:
for block in blockchain.chain:
    print(block)
    print("----------------------------")

{'index': 1, 'timestamp': 1773732818.138859, 'transactions': [], 'proof': 1, 'previous_hash': '0'}
----------------------------
{'index': 2, 'timestamp': 1773733750.3614678, 'transactions': [{'sender': '7db5e3f63b894c63', 'receiver': '28dc6f4c995d4fe9', 'amount': 25.0}], 'proof': 533, 'previous_hash': 'b4992cec8d1c2e39288f3051c177a689f8c172bc9b16865cc8c498731e52005a'}
----------------------------


In [39]:
if blockchain.is_chain_valid():
    print("Blockchain is valid.")
else:
    print("Blockchain is invalid.")

Blockchain is valid.


In [40]:
def show_all_transactions():
    cursor.execute("""
    SELECT id, sender, receiver, amount, timestamp
    FROM transactions
    ORDER BY id DESC
    """)
    
    transactions = cursor.fetchall()
    
    if transactions:
        print("All Transactions")
        print("-" * 80)
        for txn in transactions:
            print("Transaction ID :", txn[0])
            print("Sender         :", txn[1])
            print("Receiver       :", txn[2])
            print("Amount         :", txn[3])
            print("Timestamp      :", txn[4])
            print("-" * 80)
    else:
        print("No transactions found.")

In [41]:
show_all_transactions()

All Transactions
--------------------------------------------------------------------------------
Transaction ID : 1
Sender         : 7db5e3f63b894c63
Receiver       : 28dc6f4c995d4fe9
Amount         : 25.0
Timestamp      : 2026-03-17 07:48:30
--------------------------------------------------------------------------------


In [42]:
def show_user_transactions(wallet_address):
    cursor.execute("""
    SELECT id, sender, receiver, amount, timestamp
    FROM transactions
    WHERE sender = ? OR receiver = ?
    ORDER BY id DESC
    """, (wallet_address, wallet_address))
    
    transactions = cursor.fetchall()
    
    if transactions:
        print("User Transaction History")
        print("-" * 80)
        for txn in transactions:
            print("Transaction ID :", txn[0])
            print("Sender         :", txn[1])
            print("Receiver       :", txn[2])
            print("Amount         :", txn[3])
            print("Timestamp      :", txn[4])
            print("-" * 80)
    else:
        print("No transactions found for this wallet.")

In [43]:
receiver_wallet = get_wallet_by_username("rahul")
print("Rahul Wallet:", receiver_wallet)

show_user_transactions(receiver_wallet)

Rahul Wallet: 28dc6f4c995d4fe9
User Transaction History
--------------------------------------------------------------------------------
Transaction ID : 1
Sender         : 7db5e3f63b894c63
Receiver       : 28dc6f4c995d4fe9
Amount         : 25.0
Timestamp      : 2026-03-17 07:48:30
--------------------------------------------------------------------------------


In [44]:
varsha_wallet = get_wallet_by_username("varsha")
print("Varsha Wallet:", varsha_wallet)

show_user_transactions(varsha_wallet)

Varsha Wallet: 7db5e3f63b894c63
User Transaction History
--------------------------------------------------------------------------------
Transaction ID : 1
Sender         : 7db5e3f63b894c63
Receiver       : 28dc6f4c995d4fe9
Amount         : 25.0
Timestamp      : 2026-03-17 07:48:30
--------------------------------------------------------------------------------


In [45]:
varsha_wallet = get_wallet_by_username("varsha")
print("Varsha Wallet:", varsha_wallet)

show_user_transactions(varsha_wallet)

Varsha Wallet: 7db5e3f63b894c63
User Transaction History
--------------------------------------------------------------------------------
Transaction ID : 1
Sender         : 7db5e3f63b894c63
Receiver       : 28dc6f4c995d4fe9
Amount         : 25.0
Timestamp      : 2026-03-17 07:48:30
--------------------------------------------------------------------------------


In [46]:
def show_blockchain():
    print("Full Blockchain")
    print("=" * 80)
    
    for block in blockchain.chain:
        print("Block Index    :", block['index'])
        print("Timestamp      :", block['timestamp'])
        print("Transactions   :", block['transactions'])
        print("Proof          :", block['proof'])
        print("Previous Hash  :", block['previous_hash'])
        print("-" * 80)

In [47]:
show_blockchain()

Full Blockchain
Block Index    : 1
Timestamp      : 1773732818.138859
Transactions   : []
Proof          : 1
Previous Hash  : 0
--------------------------------------------------------------------------------
Block Index    : 2
Timestamp      : 1773733750.3614678
Transactions   : [{'sender': '7db5e3f63b894c63', 'receiver': '28dc6f4c995d4fe9', 'amount': 25.0}]
Proof          : 533
Previous Hash  : b4992cec8d1c2e39288f3051c177a689f8c172bc9b16865cc8c498731e52005a
--------------------------------------------------------------------------------


In [48]:
def validate_blockchain():
    if blockchain.is_chain_valid():
        print("Blockchain is valid.")
    else:
        print("Blockchain is invalid.")

In [49]:
validate_blockchain()

Blockchain is valid.
